# Solutions — Numbers, Booleans & Business Rules

Exercise notebook: `01_python_fundamentals/02_numbers_and_business_rules.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

## Exercise 1 — 🟢 Easy · Apply a percentage discount

**Working solution**

In [ ]:
def apply_discount(amount: float, pct: float) -> float:
    if amount < 0 or not 0 <= pct <= 100:
        raise ValueError(f"invalid amount={amount} or pct={pct}")
    return round(amount * (1 - pct / 100), 2)

**Explanation**

Chained comparison `0 <= pct <= 100` reads like the spec. `/` is always true division in Python 3.

**Complexity:** O(1).

**Common mistakes**

- Writing `pct >= 0 and pct <= 100` — fine, but un-idiomatic.
- Using `//` (floor division) by habit.

## Exercise 2 — 🟢 Easy / 🟡 Medium · Invoice totals with Decimal

**Working solution**

In [ ]:
from decimal import Decimal, ROUND_HALF_UP

CENT = Decimal("0.01")

def invoice_totals(lines, tax_rate="0.18"):
    subtotal = Decimal("0")
    for qty, price in lines:
        if not isinstance(qty, int) or isinstance(qty, bool) or qty <= 0:
            raise ValueError(f"invalid quantity: {qty!r}")
        subtotal += qty * Decimal(price)
    subtotal = subtotal.quantize(CENT, ROUND_HALF_UP)
    tax = (subtotal * Decimal(tax_rate)).quantize(CENT, ROUND_HALF_UP)
    return {"subtotal": subtotal, "tax": tax, "total": subtotal + tax}

**Explanation**

`Decimal` is Python's `BigDecimal`. Build it from strings, `quantize` to fix the scale. Note `bool` is a subclass
of `int` in Python (`True == 1`), so a strict check must exclude it.

**Why float is wrong**

In [ ]:
print(0.1 + 0.2)                      # 0.30000000000000004
print(sum([0.1] * 10) == 1.0)         # False
from decimal import Decimal
print(sum([Decimal("0.1")] * 10) == Decimal("1.0"))  # True

**Complexity:** O(n).

**Common mistakes**

- `Decimal(999.99)` from a float literal.
- Rounding every line's tax, then summing — totals drift from the invoice.

## Exercise 3 — 🟡 Medium · Shipping fee rule table

**Working solution**

In [ ]:
import math

RATES = {"local": (40, 10), "national": (80, 25), "international": (900, 300)}

def shipping_fee(weight_kg: float, zone: str, express: bool = False) -> int:
    try:
        base, per_kg = RATES[zone.strip().lower()]
    except KeyError:
        raise ValueError(f"unknown zone: {zone!r}") from None
    if not 0 < weight_kg <= 30:
        raise ValueError(f"weight out of range: {weight_kg}")
    extra_kg = max(0, math.ceil(weight_kg) - 1)
    fee = base + extra_kg * per_kg
    return fee * 2 if express else fee

**Explanation**

A dict lookup replaces the ladder and makes the rules editable data (you could load it from JSON).
`raise ... from None` hides the internal `KeyError` from the traceback because it's an implementation detail.

**Complexity:** O(1).

**Common mistakes**

- `int(weight)` truncates; started kilograms need `math.ceil`.
- Comparing zones without normalizing case/whitespace.

## Exercise 4 — 🟠 Hard · Transaction risk classification

**Working solution**

In [ ]:
def classify_risk(txn: dict) -> str:
    amount = txn["amount"]
    hour = txn["hour"]
    foreign = txn["country"].upper() != txn["home_country"].upper()
    if amount > 200_000 or txn.get("failed_attempts", 0) >= 3:
        return "HIGH"
    if foreign and amount > 50_000:
        return "HIGH"
    if 0 <= hour < 5 and amount > 10_000:
        return "MEDIUM"
    if txn["currency"].upper() != "INR" or foreign:
        return "MEDIUM"
    return "LOW"

**Explanation**

Early returns keep each rule flat and in priority order — no nested `else`. Underscores in numeric literals
(`200_000`) make thresholds readable. `dict.get(key, default)` handles optional fields.

**Complexity:** O(1).

**Common mistakes**

- Nesting `if/else` five levels deep.
- Recomputing `foreign` in every rule; compute derived facts once.

## Challenge — 🔴 Very Hard · Parse messy money strings

**Working solution**

In [ ]:
import re
from decimal import Decimal, InvalidOperation

CURRENCY = re.compile(r"(?i)(₹|\$|€|£|\binr\b|\busd\b|\beur\b|\bgbp\b)")

def parse_amount(text: str) -> Decimal:
    s = CURRENCY.sub("", text).replace(" ", "").strip()
    negative = s.startswith("(") and s.endswith(")")
    if negative:
        s = s[1:-1]
    if "," in s and "." in s:
        if s.rfind(",") > s.rfind("."):
            s = s.replace(".", "").replace(",", ".")
        else:
            s = s.replace(",", "")
    elif "," in s:
        if re.search(r",\d{2}$", s) and s.count(",") == 1:
            s = s.replace(",", ".")
        else:
            s = s.replace(",", "")
    if not re.fullmatch(r"[-+]?\d+(\.\d+)?", s):
        raise ValueError(f"not an amount: {text!r}")
    try:
        value = Decimal(s)
    except InvalidOperation:
        raise ValueError(f"not an amount: {text!r}") from None
    return -value if negative else value

**Explanation**

Normalize step by step: remove noise → detect sign → decide separators → validate with a strict
`fullmatch` → convert. Validating *before* `Decimal()` matters because `Decimal("NaN")` and `Decimal("1e5")` are legal.

**Complexity:** O(n).

**Common mistakes**

- `float(text.replace(',', ''))` — breaks EU formats and loses precision.
- Forgetting `Decimal('Infinity')`/`'NaN'` are accepted unless you validate first.